# 01 — Run the chart-review agent

This notebook answers one question: **what does the agent do, and what answer did it
return?** Analysis comes later.

## The agent in one picture

```text
task
  ↓
choose the next useful chart action
  ↓
call a chart tool → observe the result → update working state
  ↑                                      ↓
  └──────────────── repeat ───────────────┘
                                          ↓
                                    submit answer
```

Codex supplies the ReAct-style loop. ACR gives it a **chart-only boundary**. The agent
cannot use a shell, browser, or arbitrary filesystem access.

| Job | Tool | Plain meaning |
|---|---|---|
| Find | `list_documents` | See which notes exist. |
| Find | `search` | Find notes containing a term. |
| Read | `read` | Open one note. |
| Explain a consequential choice | `note_decision` | Record the question, choice, reason, alternatives, and claimed basis. |
| Judge evidence | `record_finding` | Say whether one note can establish the requested field. |
| Preserve proof | `record_evidence` | Save the exact supporting span. |
| Finish | `submit_answer` | Submit only after the evidence gate is satisfied. |

`note_decision` is a short audit explanation, **not private chain-of-thought**.

We use two instruction sets:

- **Task only:** asks for the diagnosis date and output format, but withholds the
  clinical evidence/conflict rules.
- **Task + policy:** adds explicit rules for what counts as evidence, which date wins,
  and what must be cited before submission.

Existing real-provider runs are reused by default, so simply reading this notebook
makes no paid call. Set `ACR_TUTORIAL_MODE=live` to run Luna again.


## Choose what to run

The live pilot is deliberately small: two paired synthetic cases × two instruction
sets × Luna. `SYN0001` has an early same-day physician diagnosis; `SYNX03` does not.
That single difference tests whether the agent handles ambiguous cytology correctly.


In [ ]:
from pathlib import Path
import asyncio
import json
import os
import subprocess

from IPython.display import Markdown, display
from acr.mvp.langtrace_io import LangtraceClient
from acr.mvp.ledger import SemanticaLedger
from acr.mvp.reconstruct import reconstruct_run
from acr.mvp.reconstruction_llm import AuditedLiteLLM
from acr.mvp.runner import run_patient

START = Path.cwd().resolve()
ROOT = START if (START / "pyproject.toml").is_file() else START.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

SEED_ROOT = ROOT / "runs/policy-experiment-20260827"
SEED_LEDGER = SEED_ROOT / "experiment-ledger.json"
LIVE_ROOT = Path(os.environ.get("ACR_TUTORIAL_RUN_ROOT", ROOT / "runs/postdoc-study"))
MODE = os.environ.get(
    "ACR_TUTORIAL_MODE", "reuse" if SEED_LEDGER.is_file() else "live"
).lower()
assert MODE in {"reuse", "live"}
EXPERIMENT_ROOT = SEED_ROOT if MODE == "reuse" else LIVE_ROOT
LEDGER_PATH = SEED_LEDGER if MODE == "reuse" else LIVE_ROOT / "ledger.json"
SPEC = ROOT / "assets/specs/STORE.390.date_of_initial_diagnosis.yaml"

PLAN = [
    {"case": case, "arm": arm, "model": "openai/gpt-5.6-luna"}
    for case in ("SYN0001", "SYNX03")
    for arm in ("task_only", "policy_bundle")
]

def short_model(value):
    return "Terra" if "terra" in str(value).lower() else "Luna"

def table(rows, columns):
    def clean(value):
        return " ".join(str(value).split()).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(clean(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

codex_version = subprocess.run(
    ["codex", "--version"], capture_output=True, text=True, check=True
).stdout.strip()
display(Markdown(f"**Mode:** `{MODE}` · **Codex:** `{codex_version}`"))
if MODE == "live":
    display(Markdown("**Live plan:**"))
    display(Markdown(table(PLAN, [
        ("case", "Case"), ("arm", "Instruction set"),
        ("model", "Review model")
    ])))
else:
    display(Markdown(
        "Loading the available historical examples with complete sealed metadata."
    ))


## Run

In live mode the collapsed cell below performs the closed loop:

```text
Luna chart review → local Langtrace → two Luna reconstructions
→ verifier agreement → selected analysis → Semantica
```

The reconstruction work is packaged here only so Notebooks 2 and 3 have something to
read. This notebook does not analyze it.


In [ ]:
if MODE == "live":
    assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY is required"
    langtrace_host = os.environ.get("LANGTRACE_API_HOST", "http://127.0.0.1:3100")
    langtrace_project = os.environ.get("LANGTRACE_PROJECT_ID", "acr_chart_review")
    langtrace_key = os.environ.get("LANGTRACE_API_KEY", "")
    client = LangtraceClient(
        api_key=langtrace_key,
        api_host=langtrace_host,
        project_id=langtrace_project,
    )
    ledger = SemanticaLedger(LEDGER_PATH)
    for item in PLAN:
        run_dir = await asyncio.to_thread(
            run_patient,
            SPEC,
            ROOT / "corpus/patients" / item["case"],
            LIVE_ROOT,
            model=item["model"],
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
            task_arm=item["arm"],
            langtrace_api_key=langtrace_key,
            langtrace_api_host=langtrace_host,
            langtrace_project_id=langtrace_project,
        )
        runner = json.loads((run_dir / "runner_meta.json").read_text())
        review = client.get_review(runner["langtrace_trace_id"])
        summary = reconstruct_run(
            review,
            ledger,
            AuditedLiteLLM(
                model="openrouter/openai/gpt-5.6-luna",
                api_key=os.environ["OPENROUTER_API_KEY"],
                temperature=0.0,
            ),
            passes=2,
            artifact_dir=run_dir / "analyses",
            reconstructor_identity="openrouter/openai/gpt-5.6-luna",
            max_attempts_per_pass=3,
        )
        assert summary["drift"]["alignment_agrees"] is True
        selected = summary["analyses"][0]["analysis_id"]
        ledger.select_analysis(
            review.run_id,
            selected,
            selected_by="postdoc-notebook-01",
            reason="Two Luna passes agreed on episode alignment.",
            provenance="DETERMINISTIC_DERIVED",
        )
else:
    assert LEDGER_PATH.is_file(), "No reusable run set was found"


## What came back?

This is intentionally the only result table in Notebook 1. Synthetic gold is shown
only because these tutorial cases were designed with a known answer.


In [ ]:
rows = []
skipped_incomplete = 0
for result_path in sorted(EXPERIMENT_ROOT.glob("*/result.json")):
    run_dir = result_path.parent
    if not (run_dir / "runner_meta.json").is_file() \
            or not (run_dir / "task_presentation.json").is_file():
        skipped_incomplete += 1
        continue
    result = json.loads(result_path.read_text())
    runner = json.loads((run_dir / "runner_meta.json").read_text())
    presentation = json.loads((run_dir / "task_presentation.json").read_text())
    case = str(result.get("patient_id") or run_dir.name.split("_", 2)[1])
    truth_path = ROOT / "corpus/patients" / case / "_ground_truth.json"
    truth = json.loads(truth_path.read_text()) if truth_path.is_file() else {}
    gold = (((truth.get("ground_truth") or {}).get(
        "STORE.390.date_of_initial_diagnosis") or {}).get("value"))
    value = result.get("value") or {}
    answer = value.get("date_of_initial_diagnosis") if isinstance(value, dict) else value
    rows.append({
        "case": case,
        "instructions": (
            "Task + policy" if presentation.get("arm_id") == "policy_bundle"
            else "Task only"
        ),
        "model": short_model(runner.get("model")),
        "answer": answer,
        "expected": gold,
        "match": "✓" if answer == gold else "✗",
    })

assert rows, "No completed result.json files were found"
display(Markdown(table(rows, [
    ("case", "Case"), ("instructions", "Instructions"), ("model", "Model"),
    ("answer", "Agent answer"), ("expected", "Synthetic gold"), ("match", "Match?")
])))
if MODE == "reuse":
    display(Markdown(
        f"**Read this correctly:** these are {len(rows)} historical runs with complete "
        "metadata, not a balanced accuracy experiment. "
        f"{skipped_incomplete} incomplete result director{'y was' if skipped_incomplete == 1 else 'ies were'} "
        "not treated as a run. Notebook 2 now follows one run step by step."
    ))
